In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
data = pd.read_csv("CKD.csv")

In [6]:
data

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [7]:
data["classification"].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [8]:
data=pd.get_dummies(data,drop_first=True)

In [9]:
data=data.astype(int)

In [10]:
data.head()

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2,76,3,0,148,57,3,137,4,12,...,0,0,0,0,0,0,1,1,0,1
1,3,76,2,0,148,22,0,137,4,10,...,1,0,0,0,0,0,1,0,0,1
2,4,76,1,0,99,23,0,138,4,12,...,1,0,0,0,0,0,1,0,0,1
3,5,76,1,0,148,16,0,138,3,8,...,1,0,0,0,0,0,1,0,1,1
4,5,50,0,0,148,25,0,137,4,11,...,1,0,0,0,0,0,1,0,0,1


In [11]:
data.isnull().sum()

age                   0
bp                    0
al                    0
su                    0
bgr                   0
bu                    0
sc                    0
sod                   0
pot                   0
hrmo                  0
pcv                   0
wc                    0
rc                    0
sg_b                  0
sg_c                  0
sg_d                  0
sg_e                  0
rbc_normal            0
pc_normal             0
pcc_present           0
ba_present            0
htn_yes               0
dm_yes                0
cad_yes               0
appet_yes             0
pe_yes                0
ane_yes               0
classification_yes    0
dtype: int64

In [12]:
indep= data.iloc[:,0:27].values

In [13]:
indep

array([[ 2, 76,  3, ...,  1,  1,  0],
       [ 3, 76,  2, ...,  1,  0,  0],
       [ 4, 76,  1, ...,  1,  0,  0],
       ...,
       [51, 70,  3, ...,  0,  0,  0],
       [51, 90,  0, ...,  1,  0,  1],
       [51, 80,  0, ...,  1,  0,  0]])

In [14]:
dep=data["classification_yes"]

In [15]:
dep

0      1
1      1
2      1
3      1
4      1
      ..
394    1
395    1
396    1
397    1
398    0
Name: classification_yes, Length: 399, dtype: int32

In [16]:
#split into training set and test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [17]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Create an untuned KNN model
knn = KNeighborsClassifier()

# Define the parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree']
}

# Create a GridSearchCV object
grid= GridSearchCV(knn, param_grid, cv=5)

# Fit the grid search object to the data
grid.fit(x_train, y_train)


GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree'],
                         'n_neighbors': [3, 5, 7],
                         'weights': ['uniform', 'distance']})

In [19]:
# Print the best parameters

print(grid.best_params_)

{'algorithm': 'auto', 'n_neighbors': 3, 'weights': 'uniform'}


In [20]:
re=grid.cv_results_

In [21]:
grid_predictions=grid.predict(x_test)

In [22]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
        

In [23]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [24]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.94      1.00      0.97        51
           1       1.00      0.96      0.98        82

    accuracy                           0.98       133
   macro avg       0.97      0.98      0.98       133
weighted avg       0.98      0.98      0.98       133



In [25]:
print(cm)

[[51  0]
 [ 3 79]]


In [26]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'algorithm': 'auto', 'n_neighbors': 3, 'weights': 'uniform'}: 0.9775556904684072


In [27]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])


0.9989239598278336

In [28]:
table=pd.DataFrame.from_dict(re)

In [29]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_algorithm,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000491,0.000982,0.094267,0.167500,auto,3,uniform,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.981481,0.981132,0.943396,0.981132,1.000000,0.977428,0.018502,1
1,0.000000,0.000000,0.011274,0.003047,auto,3,distance,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.981481,0.981132,0.943396,0.981132,1.000000,0.977428,0.018502,1
2,0.000000,0.000000,0.008941,0.001259,auto,5,uniform,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.962963,0.962264,0.924528,0.981132,0.962264,0.958630,0.018517,10
3,0.000000,0.000000,0.012679,0.001857,auto,5,distance,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.962963,0.981132,0.924528,0.981132,0.962264,0.962404,0.020671,7
4,0.000767,0.001105,0.008098,0.003066,auto,7,uniform,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.925926,0.962264,0.924528,0.981132,0.962264,0.951223,0.022320,13
5,0.001955,0.002411,0.009465,0.006876,auto,7,distance,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.925926,0.962264,0.924528,0.981132,0.962264,0.951223,0.022320,13
6,0.000931,0.001862,0.016063,0.003479,ball_tree,3,uniform,"{'algorithm': 'ball_tree', 'n_neighbors': 3, '...",0.981481,0.981132,0.943396,0.981132,1.000000,0.977428,0.018502,1
7,0.006290,0.007704,0.003546,0.006605,ball_tree,3,distance,"{'algorithm': 'ball_tree', 'n_neighbors': 3, '...",0.981481,0.981132,0.943396,0.981132,1.000000,0.977428,0.018502,1
8,0.003524,0.006098,0.009389,0.007666,ball_tree,5,uniform,"{'algorithm': 'ball_tree', 'n_neighbors': 5, '...",0.962963,0.962264,0.924528,0.981132,0.962264,0.958630,0.018517,10
9,0.000000,0.000000,0.003406,0.006812,ball_tree,5,distance,"{'algorithm': 'ball_tree', 'n_neighbors': 5, '...",0.962963,0.981132,0.924528,0.981132,0.962264,0.962404,0.020671,7


In [30]:
import pickle

In [31]:
filename="finalized_model_KNN_GridClassification.sav"

In [32]:
pickle.dump(grid ,open(filename,'wb'))

In [33]:
data.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [34]:
preinput=sc.transform([[6,82,1,95,17,0,137,3,12,38,7500,5.4,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1]])

In [35]:
loaded_model=pickle.load(open("finalized_model_KNN_GridClassification.sav",'rb'))

In [36]:
result=loaded_model.predict(preinput)

In [37]:
result

array([0])